In [1]:
!pip install langchain-openai langchain-community pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 34.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.4/331.4 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 46.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.4 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [2]:
import os
from langchain_openai import ChatOpenAI
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda
from pydantic import BaseModel, Field

In [3]:
import os
with open("/content/api_key.txt") as archivo:
  apikey = archivo.read()
os.environ["OPENAI_API_KEY"] = apikey

In [4]:
class NormaLegal(BaseModel):
    numero_ley: str = Field(description="El número de la Ley o Decreto (ej. Ley N° 31988)")
    fecha_publicacion: str = Field(description="Fecha de publicación en El Peruano (DD/MM/AAAA)")
    resumen: str = Field(description="Resumen ejecutivo de máximo 5 líneas")

In [5]:
parser = JsonOutputParser(pydantic_object=NormaLegal)

legal_prompt = ChatPromptTemplate.from_messages([
    ("system", "Eres un experto legal peruano. Extrae la información de la norma legal y devuélvela estrictamente en formato JSON.\n{format_instructions}"),
    ("user", "Analiza este extracto de El Peruano:\n{context}")
]).partial(format_instructions=parser.get_format_instructions())

# 3. Configuración del Modelo
llm = ChatOpenAI(model="gpt-4-turbo", temperature=0)

In [6]:
def load_pdf(file_path):
    loader = PyPDFLoader(file_path)
    docs = loader.load()
    # Tomamos las primeras 3 páginas para asegurar ver el encabezado y el número
    content = "\n\n".join([doc.page_content for doc in docs[:3]])
    return content

In [7]:
chain = (
    {"context": RunnableLambda(lambda x: load_pdf(x["file_path"]))}
    | legal_prompt
    | llm
    | parser # El parser ahora devuelve un dict automáticamente
)

In [8]:
file_path = "ley31814.pdf"

resultado = chain.invoke({"file_path": file_path})
resultado

{'numero_ley': 'Ley N° 31814',
 'fecha_publicacion': '05/07/2023',
 'resumen': 'La Ley N° 31814 promueve el uso de la inteligencia artificial para el desarrollo económico y social del Perú, estableciendo principios para su desarrollo seguro y ético.'}

Pdf como imagen

In [9]:
!pip install pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 43.4 MB/s eta 0:00:00


In [10]:
# 1. Instala el motor de Tesseract OCR
!apt-get install tesseract-ocr

# 2. Instala los datos de entrenamiento para español
!apt-get install tesseract-ocr-spa

# 3. Instala la librería de Python que sirve de puente (wrapper)
!pip install pytesseract

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  tesseract-ocr-spa
0 upgraded, 1 newly installed, 0 to remove and 2 not upgraded.
Need to get 951 kB of archives.
After this operation, 2,309 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 tesseract-ocr-spa all 1:4.00~git30-7274cfa-1.1 [951 kB]
Fetched 951 kB in 1s (1,004 kB/s)
Selecting previously unselected package tesseract-ocr-spa.
(Reading database ... 117540 files and directories currently installed.)
Preparing to unpack .../tesseract-ocr-spa_1%3a4.00~git30-7274cfa-1.1_all.deb ...
Unpacking tesseract-ocr-spa (1:4.00~git30-7274cfa-1.1) ...
Setting up tesseract-ocr-sp

In [11]:
!apt-get install poppler-utils

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  poppler-utils
0 upgraded, 1 newly installed, 0 to remove and 2 not upgraded.
Need to get 186 kB of archives.
After this operation, 697 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 poppler-utils amd64 22.02.0-2ubuntu0.12 [186 kB]
Fetched 186 kB in 0s (969 kB/s)
Selecting previously unselected package poppler-utils.
(Reading database ... 117544 files and directories currently installed.)
Preparing to unpack .../poppler-utils_22.02.0-2ubuntu0.12_amd64.deb ...
Unpacking poppler-utils (22.02.0-2ubuntu0.12) ...
Setting up poppler-utils (22.02.0-2ubuntu0.12) ...
Processing triggers for man-db (2.10.2-1) ...


In [12]:
!pip install pdf2image

In [13]:
import os
import pytesseract
from pdf2image import convert_from_path
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda
from pydantic import BaseModel, Field

# 1. Configuración de Tesseract en Colab
pytesseract.pytesseract.tesseract_cmd = r'/usr/bin/tesseract'

# 3. Estructura de salida deseada
class NormaLegal(BaseModel):
    numero_ley: str = Field(description="Número de la norma (ej. Ley N° 31988, D.S. N° 001-2024-TR)")
    fecha_publicacion: str = Field(description="Fecha de publicación en DD/MM/AAAA")
    resumen: str = Field(description="Resumen ejecutivo de máximo 5 líneas")

# 4. Configuración de LangChain
parser = JsonOutputParser(pydantic_object=NormaLegal)
llm = ChatOpenAI(model="gpt-4o", temperature=0)

prompt = ChatPromptTemplate.from_messages([
    ("system", "Eres un experto legal peruano especializado en El Peruano. Extrae la información solicitada y responde estrictamente en JSON.\n{format_instructions}"),
    ("user", "Analiza el siguiente texto obtenido por OCR de la norma legal:\n\n{context}")
]).partial(format_instructions=parser.get_format_instructions())

# 5. Función de carga con OCR Real
def load_pdf_with_ocr(file_path):
    print("--- Convirtiendo PDF a imagen y aplicando OCR (Tesseract) ---")
    # Convertimos solo las primeras 2 páginas (suficiente para los datos básicos)
    images = convert_from_path(file_path, first_page=1, last_page=2)

    full_text = ""
    for i, img in enumerate(images):
        # Aplicamos OCR usando el lenguaje español (spa)
        text = pytesseract.image_to_string(img, lang='spa')
        full_text += f"--- PÁGINA {i+1} ---\n{text}\n"

    if len(full_text.strip()) < 20:
        raise ValueError("No se pudo extraer texto. Verifica la calidad del PDF.")

    return full_text

# 6. Definición de la Cadena (Chain)
chain = (
    {"context": RunnableLambda(lambda x: load_pdf_with_ocr(x["file_path"]))}
    | prompt
    | llm
    | parser
)

# 7. Ejecución
if __name__ == "__main__":
    # ASEGÚRATE DE SUBIR EL ARCHIVO A COLAB PRIMERO
    file_path = "ley_imagen.pdf"

    if os.path.exists(file_path):
        try:
            resultado = chain.invoke({"file_path": file_path})
            print("\n✅ PROCESO COMPLETADO:")
            import pprint
            pprint.pprint(resultado)
        except Exception as e:
            print(f"\n❌ Error: {e}")
    else:
        print(f"Archivo '{file_path}' no encontrado. Por favor, súbelo a la carpeta de archivos de Colab.")

--- Convirtiendo PDF a imagen y aplicando OCR (Tesseract) ---

✅ PROCESO COMPLETADO:
{'fecha_publicacion': '05/07/2023',
 'numero_ley': 'Ley N° 31813',
 'resumen': 'La Ley N° 31813 declara de interés nacional la creación de la '
            'Universidad Nacional Autónoma de Cutervo en Cajamarca. El '
            'Ministerio de Educación, junto con otras entidades, coordinará '
            'acciones para su implementación, incluyendo estudios y su '
            'inclusión en el banco de proyectos de inversión.'}
